# Introduction

This case study is part of the Google Data Analytics Professional Certification. The objective is to leverage a data set from a fictional company called Bellabeat to answer a series of key business questions. In order to answer the key business questions, I will follow the steps of the data analysis process: Ask, Prepare, Process, Analyze, Share, and Act.

## Bellabeat

Bellabeat is a high-tech manufacturer of health-focused products for women. Bellabeat is a successful small company, but they have the potential to become a larger player in the global smart device market. Urška Sršen, cofounder and Chief Creative Officer of Bellabeat, believes that analyzing smart device fitness data could help unlock new growth opportunities for the company.

# 1. Ask

## 1.1 Goal

The primary objective is to analyze smart device usage data from non-Bellabeat consumers (FitBit Fitness Tracker Data) to identify global trends in health and wellness tracking. By understanding how people interact with their smart devices, the goal is to apply these insights to a specific Bellabeat product and provide high-level, data-driven recommendations to improve the company’s marketing strategy and unlock new growth opportunities in the global market

## 1.2 Stakeholders

• Urška Sršen: Bellabeat’s cofounder and Chief Creative Officer; primary requester of the analysis.

• Sando Mur: Mathematician and Bellabeat’s cofounder; a key member of the executive team.

• Bellabeat Marketing Analytics Team: The team of data analysts responsible for guiding business goals through data.

## 1.3 Scope

Analyze smart device usage to gain insights into how consumers use non-FitBit smart devices to inform a new marketing strategy.

# 2. Prepare

## 2.1 Data Integrity

The data's quality is assessed using the ROCCC framework (Reliable, Original, Comprehensive, Current, and Cited). While useful for identifying habits, the dataset has limitations regarding sample size (only 30 users), potential bias, is not current (quite old), and the timeframe is at most 31 days for each user, with one user having data only for 4 days (not Comprehensive).

## 2.2 Privacy & Ethics

The data is de-identified and all users are assigned a randomized integer.

## 2.3 Storage

The original data is pulled from Kaggle. It will then be stored between two subfolders: /raw and /processed.

# 3. Process


In [17]:
import kagglehub
import polars as pl
import os
import re

In [4]:
# Download latest version
path = kagglehub.dataset_download("arashnic/fitbit")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\jacob\.cache\kagglehub\datasets\arashnic\fitbit\versions\2


In [5]:
files = os.listdir(path)
print("Files found:", files)

Files found: ['mturkfitbit_export_3.12.16-4.11.16', 'mturkfitbit_export_4.12.16-5.12.16']


In [9]:
activity_file = os.path.join(path + "/mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/","dailyActivity_merged.csv")

In [12]:
df = pl.read_csv(activity_file, infer_schema_length=1000)

In [13]:
print(df.head())
print(df.schema)

shape: (5, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Id        ┆ ActivityD ┆ TotalStep ┆ TotalDist ┆ … ┆ FairlyAct ┆ LightlyAc ┆ Sedentary ┆ Calories │
│ ---       ┆ ate       ┆ s         ┆ ance      ┆   ┆ iveMinute ┆ tiveMinut ┆ Minutes   ┆ ---      │
│ i64       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ s         ┆ es        ┆ ---       ┆ i64      │
│           ┆ str       ┆ i64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ i64       ┆          │
│           ┆           ┆           ┆           ┆   ┆ i64       ┆ i64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 150396036 ┆ 4/12/2016 ┆ 13162     ┆ 8.5       ┆ … ┆ 13        ┆ 328       ┆ 728       ┆ 1985     │
│ 6         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 150396036 ┆ 4/13/2016 ┆ 10735     ┆ 6.97      ┆ … ┆ 19        ┆ 217       

In [18]:
def to_snake_case (name: str) -> str:
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()
    return s2.replace(" ", "_").replace("-","_")

In [19]:
new_names = {col: to_snake_case(col) for col in df.columns}
df = df.rename(new_names)
df.schema

Schema([('id', Int64),
        ('activity_date', String),
        ('total_steps', Int64),
        ('total_distance', Float64),
        ('tracker_distance', Float64),
        ('logged_activities_distance', Float64),
        ('very_active_distance', Float64),
        ('moderately_active_distance', Float64),
        ('light_active_distance', Float64),
        ('sedentary_active_distance', Float64),
        ('very_active_minutes', Int64),
        ('fairly_active_minutes', Int64),
        ('lightly_active_minutes', Int64),
        ('sedentary_minutes', Int64),
        ('calories', Int64)])

In [20]:
df = df.with_columns(pl.col("activity_date").str.to_date("%m/%d/%Y"))
print(df.head())

shape: (5, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ activity_ ┆ total_ste ┆ total_dis ┆ … ┆ fairly_ac ┆ lightly_a ┆ sedentary ┆ calories │
│ ---       ┆ date      ┆ ps        ┆ tance     ┆   ┆ tive_minu ┆ ctive_min ┆ _minutes  ┆ ---      │
│ i64       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ tes       ┆ utes      ┆ ---       ┆ i64      │
│           ┆ date      ┆ i64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ i64       ┆          │
│           ┆           ┆           ┆           ┆   ┆ i64       ┆ i64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 150396036 ┆ 2016-04-1 ┆ 13162     ┆ 8.5       ┆ … ┆ 13        ┆ 328       ┆ 728       ┆ 1985     │
│ 6         ┆ 2         ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 150396036 ┆ 2016-04-1 ┆ 10735     ┆ 6.97      ┆ … ┆ 19        ┆ 217       

In [21]:
print(df.null_count())

shape: (1, 15)
┌─────┬────────────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬──────────┐
│ id  ┆ activity_d ┆ total_step ┆ total_dist ┆ … ┆ fairly_act ┆ lightly_ac ┆ sedentary_ ┆ calories │
│ --- ┆ ate        ┆ s          ┆ ance       ┆   ┆ ive_minute ┆ tive_minut ┆ minutes    ┆ ---      │
│ u32 ┆ ---        ┆ ---        ┆ ---        ┆   ┆ s          ┆ es         ┆ ---        ┆ u32      │
│     ┆ u32        ┆ u32        ┆ u32        ┆   ┆ ---        ┆ ---        ┆ u32        ┆          │
│     ┆            ┆            ┆            ┆   ┆ u32        ┆ u32        ┆            ┆          │
╞═════╪════════════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪══════════╡
│ 0   ┆ 0          ┆ 0          ┆ 0          ┆ … ┆ 0          ┆ 0          ┆ 0          ┆ 0        │
└─────┴────────────┴────────────┴────────────┴───┴────────────┴────────────┴────────────┴──────────┘


In [22]:
df.filter(df.is_duplicated())

id,activity_date,total_steps,total_distance,tracker_distance,logged_activities_distance,very_active_distance,moderately_active_distance,light_active_distance,sedentary_active_distance,very_active_minutes,fairly_active_minutes,lightly_active_minutes,sedentary_minutes,calories
i64,date,i64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64


# 4. Analyze

In [23]:
df.describe()

statistic,id,activity_date,total_steps,total_distance,tracker_distance,logged_activities_distance,very_active_distance,moderately_active_distance,light_active_distance,sedentary_active_distance,very_active_minutes,fairly_active_minutes,lightly_active_minutes,sedentary_minutes,calories
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",940.0,"""940""",940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0
"""null_count""",0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",4.8554e9,"""2016-04-26 06:53:37.021276""",7637.910638,5.489702,5.475351,0.108171,1.502681,0.567543,3.340819,0.001606,21.164894,13.564894,192.812766,991.210638,2303.609574
"""std""",2.4248e9,null,5087.150742,3.924606,3.907276,0.619897,2.658941,0.88358,2.040655,0.007346,32.844803,19.987404,109.1747,301.267437,718.166862
"""min""",1.5040e9,"""2016-04-12""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",2.3201e9,"""2016-04-19""",3790.0,2.62,2.62,0.0,0.0,0.0,1.95,0.0,0.0,0.0,127.0,730.0,1829.0
"""50%""",4.4451e9,"""2016-04-26""",7412.0,5.25,5.25,0.0,0.21,0.24,3.37,0.0,4.0,6.0,199.0,1058.0,2135.0
"""75%""",6.9622e9,"""2016-05-04""",10725.0,7.71,7.71,0.0,2.04,0.8,4.78,0.0,32.0,19.0,264.0,1229.0,2793.0
"""max""",8.8777e9,"""2016-05-12""",36019.0,28.030001,28.030001,4.942142,21.92,6.48,10.71,0.11,210.0,143.0,518.0,1440.0,4900.0
